In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 

# Reading CSV file

In [0]:
# if the csv data is saved as a file in a volume we will use this method to read the csv file into a dataframe
df = spark.read.format('csv').option('inferSchema',True).option("header",True).load("/Volumes/workspace/pyspark_practice/pyspark_practice_volume/BigMart Sales.csv")

In [0]:
display(
    df.limit(5)
)

# Reading a table 

In [0]:
# if the data is saved as a table we will use this method lo read the table data into a dataframe
df_table = spark.table("workspace.pyspark_practice.big_mart_sales")

In [0]:
display(
    df_table.limit(5)
)

## Reading a JSON

In [0]:
df_json = spark.read.format('json').option('inferSchema',True).option("header",True).option("multiline",False).load("/Volumes/workspace/pyspark_practice/pyspark_practice_volume/drivers.json")

In [0]:
df_json.limit(5).display()

# Transformations on df

In [0]:
df.describe().display()

In [0]:
df.printSchema()

In [0]:
df.select("Item_Identifier","Item_Weight","Item_Fat_Content").limit(5).display()

In [0]:
df.select(col("Item_Identifier"),col("Item_Weight"),col('Item_Fat_Content')).limit(5).display()

In [0]:
#selecting all columns
df.select("*").limit(5).display()

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).limit(5).display()

In [0]:
# Display those rows which have "Item_Fat_Content" == "Regular"
d2 = df.filter(col("Item_Fat_Content") == "Regular").limit(5).display()

In [0]:
# Display those rows which have "Item_Weight" < 10 and "Item_Type" == "Soft Drinks"
d2 = df.filter((col("Item_Weight") < 10) & (col("Item_Type") == "Soft Drinks")).limit(5).display()

In [0]:
# Display those rows which have "Outlet_Size" is null and "Outlet_Location_Type" is either "Tier 1" or "Tier 2"
d2 = df.filter((col("Outlet_Size").isNull()) & (col("Outlet_Location_Type").isin(["Tier 1","Tier 2"]))).limit(5).display()

In [0]:
# Rename the column "Item_Weight" to "Item_Wt"
df.withColumnRenamed("Item_Weight","Item_Wt").limit(5).display()

In [0]:
# Add a new column "Stars" with values "5 Star" if "Item_Outlet_sales" >= 5000, "4 Star" if "Item_Outlet_sales" >= 4000 and "3 Star" otherwise
df.withColumn("Stars", when(col("Item_Outlet_sales") >= 5000 ,lit("5 Star")).when((col("Item_Outlet_sales") >= 4000) & (col("Item_Outlet_sales") < 5000), lit("4 Star")).otherwise(lit("3 Star"))).limit(5).display()

In [0]:
# Add a new column "Status" with values "New" if "Outlet_Establishment_Year" >= 2000 and "Old" otherwise
df.withColumn("Status", when(col("Outlet_Establishment_Year") >= 2000 ,lit("New")).otherwise(lit("Old"))).limit(5).display()

In [0]:
# Replace Regular with Reg and Low Fat with LF in the column "Item_Fat_Content"           
df.withColumn("Item_Fat_Content",regexp_replace(
    regexp_replace(col("Item_Fat_Content"),"Regular","Reg"), 
              "Low Fat",
              "LF")).limit(5).display()

In [0]:
# Use the cast function to change the data type of column Item_Weight from double to string
df.withColumn("Item_Weight",col("Item_Weight").cast("string")).limit(5).display()

In [0]:
df.sort(col("Item_MRP")).limit(5).display()

In [0]:
df.sort(col("Item_MRP").desc()).limit(5).display()

In [0]:
df.sort(["Item_Weight","Item_Visibility"], ascending = [0,0]).limit(5).display()

In [0]:
df.sort(["Item_Weight","Item_Visibility"], ascending = [0,1]).limit(5).display()

In [0]:
df.sort(col("Outlet_Size").asc_nulls_first()).limit(5).display()

In [0]:
df.drop("Item_Weight").limit(5).display()

In [0]:
df.drop("Item_Fat_Content","Item_Visibility").limit(5).display()

In [0]:
# Droping Duplictes with dropDuplicates
df.dropDuplicates().limit(5).display()


In [0]:
# Dropping Duplicates using a particular column with dropDuplicates
df.dropDuplicates(subset=["Item_Type"]).limit(5).display()

In [0]:
# Union
data = [
    ('1','Amit'),
    ('2','Aman')
]
Schema = StructType(
    [
        StructField('ID', StringType()),
        StructField('Name', StringType())

    ]
)
df1 = spark.createDataFrame(data,Schema)
df1.display()

In [0]:
data = [
    ('3','Amya'),
    ('4','Kiwi')
]

Schema = StructType(
    [
    StructField('ID', StringType()),
    StructField('Name', StringType())
    ]
)

df2 = spark.createDataFrame(data,Schema)
df2.display()

In [0]:
df1.union(df2).display()

In [0]:
data = [
    ('Amit','1'),
    ('Aman','2')
]
Schema = StructType(
    [
        StructField('Name', StringType()),
        StructField('ID', StringType())

    ]
)
df1 = spark.createDataFrame(data,Schema)
df1.display()

In [0]:
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

In [0]:
# Date Functions
df.limit(10).display()



In [0]:
df = df.withColumn("Current_date",current_date())
df.limit(10).display()

In [0]:
# date with time
df.withColumn("Date_with_Time",current_timestamp()).limit(10).display()

In [0]:
# Changing Date Formate to dd-MM-yyyy
df.withColumn("Changed_Format",date_format(col("Current_date"),'dd-MM-yyyy')).limit(10).display()

In [0]:
# Ectrating year,month from Current_date
df.withColumn("year",year(col("Current_date"))).\
    withColumn("month",month(col("Current_date"))).limit(10).display()

In [0]:
# Extrating day of the month from current_date
df.withColumn("dayofmonth", dayofmonth(col("Current_date"))).limit(10).display()

In [0]:
# Extrating day of the week from current_date (1 = Sunday,7 = Saturday)
df.withColumn("dayofweek", dayofweek(col("Current_date"))).limit(10).display()

In [0]:
# Extrating day of the year from current_date (1–366)
df.withColumn("dayofyear", dayofyear(col("Current_date"))).limit(10).display()

In [0]:
# Extracting week of the year from current_date 
df.withColumn("weekofyear",weekofyear(col("Current_date"))).limit(10).display()

In [0]:
# Adding and subtracting date to current_date
df.withColumn("date_add",date_add(col("current_date"), 10))\
    .withColumn("date_sub",date_sub(col("current_date"), 10)).limit(10).display()

In [0]:
# Differnece between two dates
df.withColumn("date_diff",date_diff(lit("2026-01-23"), col("current_date"))).limit(10).display()

In [0]:
# Handling NULLs 
df.filter(col("Outlet_Size").isNull()).limit(5).display()

In [0]:
# Droppiing Null values
df.dropna().limit(5).display()

In [0]:
# Dropping Null values using any and all
df.dropna("any").limit(10).display()
df.dropna("all").limit(10).display()

In [0]:
# Drooping Null based on specific column
df.dropna(subset = ["Outlet_Size"]).limit(10).display()

In [0]:
# Filling Null values (for string data types only)
df.fillna("NotAvailable").limit(10).display()

In [0]:
# Filling Null values for int and double data types
df.fillna(0).limit(10).display()

In [0]:
# Split
df.withColumn("Outlet_Type",split(col("Outlet_Type"), " ")).limit(10).display()

In [0]:
# Split with Indexing
df.withColumn("Outlet_Type",split(col("Outlet_Type"), " ")[0]).limit(10).display()

In [0]:
# Explode
df.withColumn("Outlet_Type", explode(split(col("Outlet_Type"), " "))).limit(10).display()
# explode_outer() - it is like explode(), but keeps rows even if array is null or empty

In [0]:
# array_contains()
df.withColumn("contains", array_contains(split(col("Outlet_Type"), " "), "Type1")).limit(10).display()

In [0]:
# Group By --> group_By
df.groupBy("Item_Type").sum("Item_Outlet_Sales").display()

#### Aggregation Functions
1. count()
2. sum()
3. avg()
4. max()
5. min()

In [0]:
# Using groupBY on multiple columns
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_Outlet_Sales")).display()

In [0]:
# Using filter() with groupBy
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_Outlet_Sales")).filter(col("sum(Item_Outlet_Sales)") > "100000").display()

In [0]:
data = [
    ('user1','book1'),
    ("user1",'book2'),
    ('user2','book2'),
    ('user2','book4'),
    ('user3','book1')
]

schema = 'user string, book string'

df_books = spark.createDataFrame(data,schema)

df_books.display()

In [0]:
df_books.groupBy('user').agg(collect_list(col('book'))).display()

In [0]:
df.limit(5).display()

In [0]:
# groupBy() with Pivot
df.groupBy("Item_Type").pivot("Outlet_Size").agg(sum("Item_Outlet_Sales").alias("Total_Sales"),avg(col("Item_Outlet_Sales").alias("Avg_Sales"))).display()

In [0]:
# when-otherwise
df.withColumn("Item_flag", when(col("Item_Type") == 'Meat', 'Non_Veg').otherwise("Veg")).limit(10).display()

In [0]:
# chained when-otherwise
df.withColumn("Veg-Non_Veg_flag", when((col("Item_Type") != "Meat") & (col("Item_MRP") > 100.0), "Veg_Expensive")\
    .when((col("Item_Type") != "Meat") & (col("Item_MRP") < 100.0), "Veg_Inexpensive")\
        .otherwise("Non_Veg")).limit(10).display()

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
# Inner join
df1.join(df2, df1.dept_id == df2.dept_id).display()

In [0]:
# Left Join
df1.join(df2, df1.dept_id == df2.dept_id, 'left').display()

In [0]:
# Right join
df1.join(df2, df1.dept_id == df2.dept_id, 'right').display()

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'anti').display()

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'full').display()

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'cross').display()

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'semi').display()